# Lab: Relationships in Space and Time

Select **two distinct datasets that each contain both spatial and temporal information**. Ask **one research question for each dataset** and answer it using the six-step reproducible analysis workflow. Create **two different types of Plotly Express figures for each question**, for a total of two research questions and four figures.

Together, each pair of figures must help answer where and when the pattern occurs. A map and a time-series plot are one useful combination; other combinations are welcome when you explain how they address both space and time.

## Learning goals

- Select and document datasets with usable location and time features.
- Ask focused questions that connect a measurement or event count to both space and time.
- Prepare dates, geographic identifiers, and summaries appropriately for the question.
- Create and compare two complementary interactive figures for each question.
- Check plotted values and ordering against the records.
- Support conclusions with evidence and explain the limitations of the data and displays.

## The reproducible analysis workflow

1. **Question:** What do we want to learn?
2. **Data:** Which observations and features can help answer it?
3. **Evidence:** What should we calculate and display to answer the question?
4. **Check:** Does the result make sense in context and agree with the original records?
5. **Conclusion:** What claim is supported?
6. **Limitation:** What should we avoid concluding?

Complete all six steps separately for each dataset. Inspect missing values during Data; use Check to investigate the actual figures and calculations. Use separate code cells for individual operations and add cells as needed.

## Using AI tools responsibly

You may use AI to ask questions, debug code, or receive feedback. You remain responsible for understanding the code, reading the source documentation, and checking every claim against your output. Complete the AI-use reflection at the end.

## Select your two datasets

Each dataset needs:

- A documented publisher and feature definitions or data dictionary.
- A usable **time feature**, such as a timestamp, date, month, or year, with observations from multiple times.
- A usable **spatial feature**, such as coordinates, station locations, or named geographic areas with matching identifiers, covering multiple places or a moving object's locations.
- A quantitative measurement with known units, or event records that support a meaningful count.
- Enough usable observations to compare places and periods. A file with dates from one fixed location alone will not support a spatial comparison.

Use two distinct datasets, not two filters of the same file. Choose your own questions and records rather than reproducing a worked lecture example. The suggestions below are starting points, not required choices. You may select another documented source.

Choose manageable geographic and time ranges before downloading. A country or city name does not automatically provide coordinates; document any station lookup, geographic-code lookup, or boundary file you need. Save fixed copies in this notebook's `data` folder, preserve the original records, and use relative paths. Include supporting lookup files with your submission.

### Candidate datasets

| Candidate and source | Useful spatial and time information | Possible question and figure pair |
| --- | --- | --- |
| [USGS earthquake catalog](https://earthquake.usgs.gov/earthquakes/search/) and [CSV field definitions](https://earthquake.usgs.gov/earthquakes/feed/v1.0/csv.php) | Event time, coordinates, magnitude, and depth. Limit the dates, geographic bounds, and magnitude range; download CSV. | Where did recorded earthquakes cluster during a selected period, and how did monthly counts change? Use a point or density map and a monthly count plot. |
| [NOAA daily weather observations](https://www.ncei.noaa.gov/cdo-web/search) and [GHCNd documentation](https://www.ncei.noaa.gov/products/land-based-station/global-historical-climatology-network-daily) | Station identifiers, dates, temperature or precipitation, and documented station locations. Select several stations and the same time span. | How did monthly precipitation differ among selected locations during a year? Use a station map of monthly summaries and a line plot or location-by-month heatmap. |
| [EPA daily air-quality data](https://www.epa.gov/outdoor-air-quality-data/download-daily-data) and [file documentation](https://aqs.epa.gov/aqsweb/airdata/download_files.html) | Monitor or county identifiers, dates, pollutant measurements or AQI. Use one pollutant and compatible units; obtain station coordinates if making a point map. | Where and when were PM2.5 concentrations highest among selected monitors? Use a map and a time-series plot grouped by monitor. |
| [NOAA Storm Events Database](https://www.ncei.noaa.gov/access/storm-events-database/) | Event dates, states/counties, event types, and locations when reported. Start with one event type, a few states, and a limited period. | How did reported hail-event counts vary across selected counties and months? Use a county choropleth and a county-by-month heatmap. County boundaries/codes are needed for the map. |

Adapt the question to the fields actually available. Counts of reported events or monitor observations can reflect reporting and sampling effort as well as the underlying phenomenon. Do not treat missing observations as zero events or average measurements with incompatible units.

## Choose two complementary figures per question

Choose **different chart types**, not two copies that differ only in color, title, filter, or static-versus-animated display. You may reuse a useful chart type with the second dataset. An animation counts as one figure, regardless of its number of frames. Animation is optional; a well-chosen static comparison can be easier to interpret.

| Figure | Plotly Express tools | What it can help you see |
| --- | --- | --- |
| Time-series line plot | `px.line()` | How a measurement changes over time; color or separate panels can distinguish locations. Sort dates within each series before connecting points. |
| Point or bubble map | `px.scatter_geo()` or `px.scatter_map()` | Where observations occur; color and size can show quantitative values. Time can appear in hover information, filters, or animation. |
| Choropleth | `px.choropleth()` | How an area-level summary varies geographically, optionally animated by period. Match country/state codes or supply matching GeoJSON boundaries. |
| Bubble scatterplot | `px.scatter()` | How two quantitative features relate; geographic groups and time can be represented by color and animation. Explain how location is represented if this replaces a map. |
| Geographic path — an additional option | `px.line_geo()` or `px.line_map()` | A recorded route or moving object's sequence of positions. Sort within each tracked object; do not connect unrelated events into a path. |
| Spatial density map — an additional option | `px.density_map()` | Where many nearby point observations concentrate. The smoothing radius affects the appearance; this is not automatically a rate or a map of risk. |
| Location-by-time heatmap — an additional option | `px.imshow()` | Rows represent locations, columns represent ordered periods, and color represents a count or measurement. Summarize to one value per location-period first. |

Use the [Plotly map guide](https://plotly.com/python/map-configuration/), [heatmap guide](https://plotly.com/python/heatmaps/), and [animation guide](https://plotly.com/python/animations/) for examples and parameters. Tile maps need access to background tiles; geographic maps may also need internet access for boundary outlines.

### Starting patterns for additional chart types

These are **adaptation examples**, not code to run unchanged. Replace the DataFrame and feature names with yours. They do not add requirements to your four figures.

**Path of a tracked object:** only use this when consecutive rows describe the same object's movement.

```python
track_df.sort_values(['object_id', 'date'], inplace=True)
fig = px.line_geo(track_df, lat='latitude', lon='longitude',
                  color='object_id', hover_data=['date'],
                  title='Recorded paths')
fig.update_layout(height=500)
fig.show()
```

**Density of point observations:** omit `z` to give every row equal weight. Document any deliberate weighting.

```python
fig = px.density_map(events_df, lat='latitude', lon='longitude',
                     radius=15, map_style='open-street-map',
                     title='Concentration of recorded events')
fig.update_layout(height=500)
fig.show()
```

**Location-by-month heatmap:** this example uses a mean measurement. Use a sum or count only when that statistic answers your question. Missing cells remain missing unless you can establish that a zero is appropriate.

```python
observations_df['month'] = observations_df['date'].dt.to_period('M').astype(str)
monthly_df = observations_df.groupby(['location', 'month'])['value'].mean().reset_index()
grid = monthly_df.pivot(index='location', columns='month', values='value')
grid = grid.reindex(sorted(grid.columns), axis=1)
fig = px.imshow(grid, aspect='auto',
                labels={'x':'Month', 'y':'Location', 'color':'Mean value'},
                title='Mean measurement by location and month')
fig.update_layout(height=500)
fig.show()
```

### Make the comparison interpretable

Use descriptive titles, labels with units, and informative hover fields. Start with `fig.update_layout(height=500)` and adjust only if needed. Keep scales consistent when comparing frames or panels. Use a manageable number of time periods—monthly, quarterly, or yearly frames often work better than hundreds of daily frames. Explain any aggregation and retain the original dates.

For maps, confirm that geographic codes match the boundaries and that points appear in plausible places. Numeric identifiers are not quantitative measurements. A colored area or large marker does not necessarily mean greater risk: consider population, area, observation effort, and what the encoded value actually measures.

In [1]:
# Libraries for working with data and interactive figures
from pathlib import Path
import pandas as pd
import plotly.express as px
from IPython.display import display

# Keep exported interactive charts next to the notebook
Path("figures").mkdir(exist_ok=True)

## Investigation 1: Your dataset and research question

### Step 1: Question

Ask one focused question that depends on **both where and when** something occurs. Name the measurement or count, the places, and the period. Record a prediction before producing the figures.

**What is your research question, and what do you predict? Why is it useful to consider both space and time?**

**Research question:** How did the percentage of Florida red tide samples classified as **High** change by month and sampled location between June and November 2018?

**Prediction (recorded as a proposed expectation, not a statistical result):** High-abundance samples will be clustered at some Florida coastal sampling sites rather than spread evenly, and the percentage of sampled observations classified as High will vary by month. Dates show *when* observed abundance changed; mapped coordinates show *where* samples were collected.

### Step 2: Data

Document the source and explain what one row represents. Load a fixed copy, inspect the first and last rows and shape, use `.info()` to inspect the data storage types and non-missing values, and count missing values. Parse dates when needed and preserve geographic identifiers appropriately.

**Give the dataset title, publisher, direct source URL, dictionary reference, and access date. Identify your saved file and any lookup or boundary files.**

**Dataset:** *Florida Red Tide Samples, June–November 2018* (fixed course-provided CSV). **Publisher and background:** Florida Fish and Wildlife Conservation Commission (FWC), Fish and Wildlife Research Institute (FWRI), Harmful Algal Bloom Monitoring Database. **Data source and sampling context:** https://myfwc.com/research/redtide/monitoring/database/ . **Feature and measurement reference:** https://myfwc.com/research/redtide/tools/ . **Saved file:** `data/florida_red_tide_samples_june_november_2018.csv` supplied in the course repository. **Access/review date:** October 19, 2026. The course CSV is a fixed extract; this analysis does not claim that the public database page downloads the identical historical subset. No boundary or geographic lookup file is required because the file includes latitude and longitude.

**What does one row represent? Which places and dates are covered, and how were the observations collected?**

One original row records one dated sampling observation at a named Florida location, with the coordinates, sample depth where available, *Karenia brevis* cell concentration (cells/L), and an abundance category. The supplied fixed extract has 7,292 records collected from June 1 to November 30, 2018, at locations throughout Florida's sampled waters. These are measurements collected at sampling sites by monitoring partners, not continuous coverage of all surrounding water. Several visits to one location are separate records.

In [2]:
# Load the fixed course CSV; convert dates so we can sort/group months
red_tide_df = pd.read_csv(
    "data/florida_red_tide_samples_june_november_2018.csv",
    parse_dates=["sample_date"]
)
print("Florida red tide dataset loaded successfully.")

Florida red tide dataset loaded successfully.


In [3]:
# Inspect the beginning of the source file
red_tide_df.head()

,sample_date,location,latitude,longitude,sample_depth_m,karenia_brevis_cells_per_liter,abundance_category
0,2018-06-01,Red Reef Park Beach,26.36043,-80.06847,0.5,0.0,Not present/background
1,2018-06-01,Gumbo Limbo Nature Center (Lake Wyman),26.36751,-80.07118,0.5,0.0,Not present/background
2,2018-06-01,Bay Dock (Sarasota Bay),27.33160,-82.57790,0.5,5000.0,Very low
3,2018-06-01,New Pass Dock (Sarasota Bay),27.33375,-82.57938,0.5,1000.0,Not present/background
4,2018-06-01,Dixie Highway Bridge; E of (Indian River),27.85566,-80.49085,1.3,0.0,Not present/background


In [4]:
# Inspect the final five source rows
red_tide_df.tail()

,sample_date,location,latitude,longitude,sample_depth_m,karenia_brevis_cells_per_liter,abundance_category
7287,2018-11-29,Brooks Bridge; W of (Santa Rosa Sound),30.400800,-86.600800,0.1,0.0,Not present/background
7288,2018-11-30,Gasparilla Island; 10 mi W of,26.722583,-82.366933,0.5,20667.0,Low
7289,2018-11-30,Gasparilla Island; 13 mi W of,26.760700,-82.475550,0.5,391020.0,Medium
7290,2018-11-30,Bay Dock (Sarasota Bay),27.331600,-82.577900,0.5,534000.0,Medium
7291,2018-11-30,New Pass Dock (Sarasota Bay),27.333750,-82.579380,0.5,274000.0,Medium


In [5]:
# Confirm source row and column counts
red_tide_df.shape

(7292, 7)

In [6]:
# Inspect types and count missing values, including unused fields
red_tide_df.info()
print("\nMissing values by feature:")
print(red_tide_df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7292 entries, 0 to 7291
Data columns (total 7 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   sample_date                     7292 non-null   datetime64[ns]
 1   location                        7292 non-null   object        
 2   latitude                        7292 non-null   float64       
 3   longitude                       7292 non-null   float64       
 4   sample_depth_m                  7259 non-null   float64       
 5   karenia_brevis_cells_per_liter  7292 non-null   float64       
 6   abundance_category              7292 non-null   object        
dtypes: datetime64[ns](1), float64(4), object(2)
memory usage: 398.9+ KB

Missing values by feature:
sample_date                        0
location                           0
latitude                           0
longitude                          0
sample_depth_m                   

**Complete the table for every feature used in the question, preparation, or figures. Add rows as needed.**

| Feature | Meaning and units or codes | Feature type | Data storage type | Missing values |
| --- | --- | --- | --- | --- |
| sample_date | Date of collected sample | Temporal | datetime64[ns] | 0 |
| location | Named sampling location | Categorical | object | 0 |
| latitude | Latitude, decimal degrees | Quantitative | float64 | 0 |
| longitude | Longitude, decimal degrees | Quantitative | float64 | 0 |
| sample_depth_m | Sampling depth, metres (inspected; not modeled) | Quantitative | float64 | 33 |
| karenia_brevis_cells_per_liter | Estimated concentration, cells/L | Quantitative | float64 | 0 |
| abundance_category | FWC-style category: Not present/background, Very low, Low, Medium, High | Categorical | object | 0 |
| month | Derived month, YYYY-MM | Temporal | object (string) | 0 |
| is_high | Derived: category equals High (True/False) | Categorical | bool | 0 |
| longitude_band | Approximate split at 81.5°W (not an official coast boundary) | Categorical | object | 0 |

Use quantitative, categorical, or temporal for feature type.

**Which observations and features will you use? Explain your time periods, geographic grouping, filtering, missing-value decisions, and any counts, means, sums, or rates.**

I retained the fixed June–November 2018 sample records with usable dates, coordinates, and abundance categories. I created monthly periods and the indicator `is_high` (`abundance_category == "High"`). I computed total observations, High observations, and High percentages each month. To support a simple geographical comparison, I also grouped sample **longitudes** into sites west of 81.5°W and sites at or east of 81.5°W. This is an approximate numerical split, not a coastline or official ecological region. The animated map uses all individual sample coordinates and dates; the line chart uses statewide monthly percentages. I preserved all 7,292 records; 33 missing sample depths do not affect any plotted measurement.

In [7]:
# Preserve original observations, in chronological order
red_tide_analysis = red_tide_df.sort_values("sample_date").copy()
red_tide_analysis["month"] = red_tide_analysis["sample_date"].dt.strftime("%Y-%m")
red_tide_analysis["is_high"] = red_tide_analysis["abundance_category"].eq("High")
red_tide_analysis["high_status"] = red_tide_analysis["is_high"].map(
    {True: "High", False: "Other"}
)

# Approximate longitude regions to compare how observations vary by location
red_tide_analysis["longitude_band"] = red_tide_analysis["longitude"].apply(
    lambda x: "West of 81.5°W" if x <= -81.5 else "81.5°W or east"
)

# Count sample observations and compute percentages (not percentage of ocean area)
monthly_summary = (
    red_tide_analysis.groupby("month")
    .agg(total_samples=("is_high", "size"), high_samples=("is_high", "sum"))
    .reset_index()
    .sort_values("month")
)
monthly_summary["high_percentage"] = (
    monthly_summary["high_samples"] / monthly_summary["total_samples"] * 100
).round(1)

regional_summary = (
    red_tide_analysis.groupby(["month", "longitude_band"])
    .agg(total_samples=("is_high", "size"), high_samples=("is_high", "sum"))
    .reset_index()
    .sort_values(["month", "longitude_band"])
)
regional_summary["high_percentage"] = (
    regional_summary["high_samples"] / regional_summary["total_samples"] * 100
).round(1)

print("Original records:", len(red_tide_df))
print("Retained records:", len(red_tide_analysis))
print("Removed records:", len(red_tide_df) - len(red_tide_analysis))
print("Unmatched geographic identifiers: none (coordinates are directly available)")
print("\nCategory counts:")
print(red_tide_analysis["abundance_category"].value_counts())
print("\nStatewide monthly results:")
display(monthly_summary)
print("\nApproximate longitude-region results:")
display(regional_summary)

Original records: 7292
Retained records: 7292
Removed records: 0
Unmatched geographic identifiers: none (coordinates are directly available)

Category counts:
abundance_category
Not present/background    4558
Medium                     863
Low                        708
Very low                   603
High                       560
Name: count, dtype: int64

Statewide monthly results:


,month,total_samples,high_samples,high_percentage
0,2018-06,991,19,1.9
1,2018-07,865,62,7.2
2,2018-08,1236,191,15.5
3,2018-09,1196,76,6.4
4,2018-10,1855,95,5.1
5,2018-11,1149,117,10.2



Approximate longitude-region results:


,month,longitude_band,total_samples,high_samples,high_percentage
0,2018-06,81.5°W or east,205,0,0.0
1,2018-06,West of 81.5°W,786,19,2.4
2,2018-07,81.5°W or east,142,0,0.0
3,2018-07,West of 81.5°W,723,62,8.6
4,2018-08,81.5°W or east,214,0,0.0
5,2018-08,West of 81.5°W,1022,191,18.7
6,2018-09,81.5°W or east,169,0,0.0
7,2018-09,West of 81.5°W,1027,76,7.4
8,2018-10,81.5°W or east,828,23,2.8
9,2018-10,West of 81.5°W,1027,72,7.0


**How many records were removed from the dataset? Explain why, and state what one row represents after any aggregation.**

**7,292 original records, 7,292 retained, and 0 removed.** The only missing field is sample depth (33 values), which is not needed for either figure or for calculating High-sample percentages. The unaggregated and mapped data have one row per collected sample, which means repeated observations at a site remain distinct. The statewide monthly summary has one row per **month** (six rows), with a sample count, High count, and High percentage. The region-by-month summary has one row per **month × approximate longitude group** (12 rows). All mapped records have valid coordinates, so no geographic identifiers required matching.

### Step 3: Evidence

Create two different Plotly Express figure types that answer this same question. Together they must display the spatial and temporal patterns. Use the prepared data consistently and explain any difference in coverage between the figures.

**Which two chart types will you use, and what part of the question should each make easier to answer? Specify the time, location, and quantitative encodings.**

**Figure 1 — line chart:** Month (June–November 2018) on x, percentage of Florida samples classified High on y; hover gives High count and total sample count. This highlights *when* the reported High category was most common among samples across the state. **Figure 2 — animated geographic point map:** Latitude and longitude locate each sample, dot color is High versus Other, monthly frames give time, and hover reports sampling site, date, abundance category, and concentration in **cells/L**. This highlights *where* the samples came from. The line aggregates all observations; the map shows the original records, so busy/coincident dots can overlap.

In [8]:
# Figure 1 — a monthly rate among sampled observations (not a map of affected water)
fig1 = px.line(
    monthly_summary,
    x="month", y="high_percentage", markers=True,
    title="Florida red tide: High-abundance samples by month (June–November 2018)",
    labels={"month": "Month", "high_percentage": "High-abundance samples (%)"},
    hover_data={"high_samples": True, "total_samples": True}
)
fig1.update_layout(width=900, height=500)
fig1.write_html("figures/florida_red_tide_monthly.html", include_plotlyjs=True)
fig1.show()

**What does Figure 1 show? Describe a pattern and record at least one value with its place, time, and units or count.**

Figure 1 shows that the share of **sampled observations** labeled High changed over time across sampled **Florida** locations. It increased from **1.9% in June 2018** (19 of 991 samples) to **15.5% in August 2018** (191 of 1,236), then declined to **5.1% in October** (95 of 1,855) and rebounded to **10.2% in November** (117 of 1,149). The y-axis is a percentage of collected samples, not the proportion of all Florida waters experiencing red tide.

In [9]:
# Figure 2 — individual site measurements in month-by-month map frames
# New Plotly function px.scatter_map avoids the older scatter_mapbox warning.
fig2 = px.scatter_map(
    red_tide_analysis,
    lat="latitude", lon="longitude", color="high_status",
    animation_frame="month", hover_name="location",
    hover_data={"sample_date": True, "abundance_category": True,
                "karenia_brevis_cells_per_liter": ":,.0f",
                "latitude": ":.4f", "longitude": ":.4f", "high_status": False},
    center={"lat": 27.7, "lon": -82.0}, zoom=5.5,
    map_style="open-street-map",
    title="Florida red tide sampling locations (June–November 2018)",
    labels={"high_status": "Abundance group",
            "karenia_brevis_cells_per_liter": "K. brevis (cells/L)"},
    color_discrete_map={"High": "#c92332", "Other": "#2366b7"},
    category_orders={"month": sorted(red_tide_analysis["month"].unique())}
)
fig2.update_layout(width=950, height=650)
fig2.write_html("figures/florida_red_tide_map.html", include_plotlyjs=True)
fig2.show()

**What does Figure 2 add? Record at least one value with its place, time, and units or count, and compare its message with Figure 1.**

Figure 2 adds the **specific places** behind Figure 1's statewide monthly rate. For example, a High sample at **Englewood Beach, Florida**, on **August 1, 2018**, measured **7,285,740 *K. brevis* cells/L** (26.92467°N, 82.36189°W). The monthly summary confirms August had 191 High observations among 1,236 statewide samples (15.5%); the map identifies where particular observations were taken. In the simple geographic split, August had **191 High among 1,022 samples (18.7%)** west of 81.5°W, and **0 among 214 sampled observations** at or east of it. Map colors indicate category at sampled points, not a continuously measured bloom boundary.

### Step 4: Check

Use hover to compare at least two plotted values with the source records or a calculated summary. Check chronological order, location matching, and one unusual high, low, or apparent gap that matters to your question. Explain any problem you corrected. Do not repeat the initial missing-value count.

In [10]:
# Check two actual mapped values against individual source rows
check_june = red_tide_df.loc[
    (red_tide_df["location"] == "Blind Pass Beach") &
    (red_tide_df["sample_date"] == pd.Timestamp("2018-06-11")) &
    (red_tide_df["karenia_brevis_cells_per_liter"] == 2913000)
]
check_aug = red_tide_df.loc[
    (red_tide_df["location"] == "Englewood Beach") &
    (red_tide_df["sample_date"] == pd.Timestamp("2018-08-01")) &
    (red_tide_df["karenia_brevis_cells_per_liter"] == 7285740)
]
assert len(check_june) == 1 and len(check_aug) == 1
assert check_june.iloc[0]["abundance_category"] == "High"
assert check_aug.iloc[0]["abundance_category"] == "High"

# Check the monthly figure is consistent with source records
aug_source = red_tide_analysis.loc[red_tide_analysis["month"] == "2018-08"]
assert len(aug_source) == 1236 and int(aug_source["is_high"].sum()) == 191
assert list(monthly_summary["month"]) == sorted(monthly_summary["month"])
assert red_tide_analysis[["latitude", "longitude"]].notna().all().all()
print("Verified June site:", check_june[["sample_date", "location", "latitude", "longitude", "karenia_brevis_cells_per_liter"]].to_string(index=False))
print("Verified August site:", check_aug[["sample_date", "location", "latitude", "longitude", "karenia_brevis_cells_per_liter"]].to_string(index=False))
print("\nAugust: 191 of 1,236 =", round(191 / 1236 * 100, 1), "%")
print("Date order correct; all source coordinates present.")

Verified June site: sample_date         location  latitude  longitude  karenia_brevis_cells_per_liter
 2018-06-11 Blind Pass Beach  26.96324  -82.38565                       2913000.0
Verified August site: sample_date        location  latitude  longitude  karenia_brevis_cells_per_liter
 2018-08-01 Englewood Beach  26.92467  -82.36189                       7285740.0

August: 191 of 1,236 = 15.5 %
Date order correct; all source coordinates present.


**Which two values did you check, and did they agree? What did your check establish about order, location, or an unusual pattern?**

I checked **Blind Pass Beach on June 11, 2018** (**2,913,000 cells/L**, High) and **Englewood Beach on August 1, 2018** (**7,285,740 cells/L**, High) against exact location, date, concentration, and coordinate values in the original CSV. Both checks agree with the intended map hover values. A separate calculation verified Figure 1's August point, **191 ÷ 1,236 × 100 = 15.5%**. The code asserts that monthly labels are sorted June–November and that coordinates exist. The August spike represents a change in *sample proportions*, not proof that the bloom covered a larger geographic area.

### Step 5: Conclusion

Restate your research question, then answer it using evidence from **both** figures. Include a comparison across places and a comparison across times, with values and units. Explain whether the evidence supports your prediction.

**Restate your question and write your conclusion.**

**Question:** How did the percentage of sampled Florida red tide observations labeled High change by month and location during June–November 2018? **Answer:** Among Florida sample observations, the High percentage rose from **1.9% in June** (19/991) to **15.5% in August** (191/1,236), fell to **5.1% in October** (95/1,855), and reached **10.2% in November** (117/1,149). The map shows those High records were not evenly distributed across sampled coordinates. In our approximate longitude comparison, **537/5,367 (10.0%)** observations west of 81.5°W were High, versus **23/1,925 (1.2%)** at or east of 81.5°W. A verified **Englewood Beach** observation on August 1 measured **7,285,740 cells/L**. This supports the prediction of location and month differences **among sampled records**, but it does not establish unsampled coastal conditions.

### Step 6: Limitation

Identify **two limitations of your analysis**. Then explain which aspects of the research question are easier or harder to answer from each figure.

**What are two limitations, and how do the strengths and weaknesses of your figures affect the answer?**

**Two limitations:** (1) Sampling effort varies across months and locations; these percentages describe **collected samples**, not the proportion of Florida water experiencing blooms or changes in true statewide prevalence. FWC notes that many monitoring samples are collected in response to events. (2) The simple west/east longitude cut is **not an ecological or coastline boundary**, and multiple readings at one site cause overlapping map markers. The line chart is best for comparing statewide monthly rates but hides location variation; the map is best for inspecting sampled sites and concentrations but hides overlapping records and is not a representative area-wide surface. Neither display establishes why abundance changed.

## Investigation 2: Your dataset and research question

### Step 1: Question

Ask one focused question that depends on **both where and when** something occurs. Name the measurement or count, the places, and the period. Record a prediction before producing the figures.

**What is your research question, and what do you predict? Why is it useful to consider both space and time?**

**Research question:** From **2020 through 2025**, how did the percentage of Atlantic tropical cyclone **best-track observations** with maximum sustained winds of at least **64 knots** differ by year between locations **west of 65°W** and locations **at or east of 65°W**?

**Prediction:** The recorded share of observations meeting the hurricane-force threshold will differ by year and by longitude group rather than staying constant. Dates identify when high-wind positions were recorded, while coordinates show where the observed cyclone centers were located. This is a different multi-storm, multi-year comparison from the lecture's single-storm Hurricane Ian demonstration.

### Step 2: Data

Document the source and explain what one row represents. Load a fixed copy, inspect the first and last rows and shape, use `.info()` to inspect the data storage types and non-missing values, and count missing values. Parse dates when needed and preserve geographic identifiers appropriately.

**Give the dataset title, publisher, direct source URL, dictionary reference, and access date. Identify your saved file and any lookup or boundary files.**

**Dataset:** *Atlantic Tropical Cyclone Observations, 2020–2025*, a fixed course-provided HURDAT2-derived CSV. **Underlying publisher:** National Oceanic and Atmospheric Administration (NOAA), National Hurricane Center (NHC). **Direct publisher archive:** https://www.nhc.noaa.gov/data/#hurdat . **Dictionary and data format:** https://www.nhc.noaa.gov/data/hurdat/hurdat2-format-nov2019.pdf . **Saved file:** `../lecture/data/atlantic_tropical_cyclone_observations_2020_2025.csv` (referenced directly from `week03/lecture/data/` in the supplied course project). **Access/review date:** October 19, 2026. This notebook analyzes the **fixed course extract**, not the entire current official archive. No geography lookup is required, because cyclone center positions include numeric latitude and longitude.

**What does one row represent? Which places and dates are covered, and how were the observations collected?**

Each row is one archived **best-track position and intensity observation** for an Atlantic tropical or subtropical cyclone: storm identifier, storm name, UTC observation timestamp, status, center coordinates, sustained wind (knots), and central pressure (millibars). This course extract contains **2,530 observations** for **120 storm identifiers** during 2020–2025; the observed center positions cover Atlantic basin waters including the Caribbean and Gulf. Best-track entries are assembled from post-storm scientific assessments, generally at six-hour intervals with additional event entries. They are not a random sample of weather across the entire ocean. Repeated observations of the same storm are distinct positions.

In [11]:
# Load and chronologically parse the second, independent fixed dataset
cyclone_df = pd.read_csv(
    "../lecture/data/atlantic_tropical_cyclone_observations_2020_2025.csv",
    parse_dates=["observation_time"]
)
print("Atlantic cyclone best-track dataset loaded.")

Atlantic cyclone best-track dataset loaded.


In [12]:
# Beginning of source file (storm IDs are treated as identifiers, not numeric values)
cyclone_df.head()

,storm_id,storm_name,observation_time,status,record_identifier,latitude,longitude,maximum_sustained_wind_knots,minimum_central_pressure_mb
0,AL012020,Arthur,2020-05-16 18:00:00,TD,NaN,28.0,-78.7,30,1008
1,AL012020,Arthur,2020-05-17 00:00:00,TS,NaN,28.9,-78.0,35,1006
2,AL012020,Arthur,2020-05-17 06:00:00,TS,NaN,29.6,-77.6,35,1004
3,AL012020,Arthur,2020-05-17 12:00:00,TS,NaN,30.3,-77.5,35,1003
4,AL012020,Arthur,2020-05-17 18:00:00,TS,NaN,31.0,-77.3,40,1003


In [13]:
# Inspect the final five source rows
cyclone_df.tail()

,storm_id,storm_name,observation_time,status,record_identifier,latitude,longitude,maximum_sustained_wind_knots,minimum_central_pressure_mb
2525,AL132025,Melissa,2025-10-30 06:00:00,HU,NaN,25.0,-73.8,85,968
2526,AL132025,Melissa,2025-10-30 12:00:00,HU,NaN,26.7,-72.7,90,965
2527,AL132025,Melissa,2025-10-30 18:00:00,HU,NaN,28.9,-70.9,85,967
2528,AL132025,Melissa,2025-10-31 00:00:00,HU,NaN,31.3,-68.7,80,970
2529,AL132025,Melissa,2025-10-31 06:00:00,HU,NaN,34.5,-65.5,75,972


In [14]:
# Confirm source row and column counts
cyclone_df.shape

(2530, 9)

In [15]:
# Inspect types, non-missing values, and missing values in all source columns
cyclone_df.info()
print("\nMissing values by feature:")
print(cyclone_df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2530 entries, 0 to 2529
Data columns (total 9 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   storm_id                      2530 non-null   object        
 1   storm_name                    2530 non-null   object        
 2   observation_time              2530 non-null   datetime64[ns]
 3   status                        2530 non-null   object        
 4   record_identifier             122 non-null    object        
 5   latitude                      2530 non-null   float64       
 6   longitude                     2530 non-null   float64       
 7   maximum_sustained_wind_knots  2530 non-null   int64         
 8   minimum_central_pressure_mb   2530 non-null   int64         
dtypes: datetime64[ns](1), float64(2), int64(2), object(4)
memory usage: 178.0+ KB

Missing values by feature:
storm_id                           0
storm_name          

**Complete the table for every feature used in the question, preparation, or figures. Add rows as needed.**

| Feature | Meaning and units or codes | Feature type | Data storage type | Missing values |
| --- | --- | --- | --- | --- |
| storm_id | Official tracked system identifier (e.g. AL012020) | Categorical | object | 0 |
| storm_name | Recorded storm name | Categorical | object | 0 |
| observation_time | Best-track timestamp (UTC) | Temporal | datetime64[ns] | 0 |
| status | TD, TS, HU, SS, or SD storm status | Categorical | object | 0 |
| record_identifier | Event marker (blank for no special marker); inspected but not analyzed | Categorical | object | 2408 |
| latitude | Cyclone center latitude, decimal degrees | Quantitative | float64 | 0 |
| longitude | Cyclone center longitude, decimal degrees | Quantitative | float64 | 0 |
| maximum_sustained_wind_knots | Maximum sustained wind speed in knots | Quantitative | int64 | 0 |
| minimum_central_pressure_mb | Central pressure in millibars (hover only) | Quantitative | int64 | 0 |
| year | Calendar year derived from observation time | Temporal | int32 | 0 |
| longitude_band | West of 65°W vs. 65°W or east, a simple spatial grouping | Categorical | object | 0 |
| hurricane_force | Wind ≥ 64 knots (boolean indicator) | Categorical | bool | 0 |

Use quantitative, categorical, or temporal for feature type.

**Which observations and features will you use? Explain your time periods, geographic grouping, filtering, missing-value decisions, and any counts, means, sums, or rates.**

I analyzed the entire fixed 2020–2025 course CSV without removing storms or changing recorded winds. For every observation, I derived the calendar year, defined `hurricane_force` as **maximum sustained wind ≥ 64 knots**, and grouped center longitude into **west of 65°W** versus **65°W or east**. The line chart reports the **share of recorded best-track positions** meeting this threshold in each year/longitude group. The animated map plots each observation at its recorded center coordinates and shows year and wind category. The missing `record_identifier` values simply mean no special event marker, and that optional feature is not needed for the analysis. Storm IDs remain strings. Because repeated positions of one storm are counted separately, percentages describe observed positions, **not percentages of storms** or ocean area.

In [16]:
# Prepare time and spatial categories without discarding any rows
cyclone_analysis = cyclone_df.sort_values(
    ["observation_time", "storm_id"]
).copy()
cyclone_analysis["year"] = cyclone_analysis["observation_time"].dt.year
cyclone_analysis["longitude_band"] = cyclone_analysis["longitude"].apply(
    lambda x: "West of 65°W" if x < -65 else "65°W or east"
)
cyclone_analysis["hurricane_force"] = (
    cyclone_analysis["maximum_sustained_wind_knots"] >= 64
)
cyclone_analysis["wind_group"] = cyclone_analysis["hurricane_force"].map(
    {True: "At least 64 kt", False: "Below 64 kt"}
)

# One annual percentage per longitude group; do not confuse records with storms
cyclone_summary = (
    cyclone_analysis.groupby(["year", "longitude_band"])
    .agg(total_positions=("hurricane_force", "size"),
         hurricane_force_positions=("hurricane_force", "sum"))
    .reset_index()
    .sort_values(["longitude_band", "year"])
)
cyclone_summary["force_percentage"] = (
    100 * cyclone_summary["hurricane_force_positions"]
    / cyclone_summary["total_positions"]
).round(1)

print("Original records:", len(cyclone_df))
print("Retained records:", len(cyclone_analysis))
print("Removed records:", len(cyclone_df) - len(cyclone_analysis))
print("Unmatched location IDs: none (latitude/longitude already included)")
print("Number of storm IDs:", cyclone_analysis["storm_id"].nunique())
print("Missing special event identifiers:", cyclone_df["record_identifier"].isna().sum())
display(cyclone_summary)

Original records: 2530
Retained records: 2530
Removed records: 0
Unmatched location IDs: none (latitude/longitude already included)
Number of storm IDs: 120
Missing special event identifiers: 2408


,year,longitude_band,total_positions,hurricane_force_positions,force_percentage
0,2020,65°W or east,288,63,21.9
2,2021,65°W or east,237,84,35.4
4,2022,65°W or east,117,34,29.1
6,2023,65°W or east,364,86,23.6
8,2024,65°W or east,180,74,41.1
10,2025,65°W or east,140,36,25.7
1,2020,West of 65°W,373,93,24.9
3,2021,West of 65°W,189,33,17.5
5,2022,West of 65°W,198,85,42.9
7,2023,West of 65°W,124,43,34.7


**How many records were removed from the dataset? Explain why, and state what one row represents after any aggregation.**

**2,530 original best-track position records, 2,530 retained, 0 removed.** There were no missing observation times, winds, latitudes, or longitudes. The `record_identifier` field was blank for 2,408 records, but it is only a special-event marker and is not used in the analysis. No lookup mismatch occurred, because the dataset already provides coordinates. Before aggregation, each row means **one storm center observation at a UTC time**, so a storm can contribute several rows. After aggregation, each summary row means **one calendar year × longitude group**, containing a count of best-track positions, a count with wind ≥64 knots, and that group's percentage (12 rows total).

### Step 3: Evidence

Create two different Plotly Express figure types that answer this same question. Together they must display the spatial and temporal patterns. Use the prepared data consistently and explain any difference in coverage between the figures.

**Which two chart types will you use, and what part of the question should each make easier to answer? Specify the time, location, and quantitative encodings.**

**Figure 3 — line chart:** Years 2020–2025 on the x-axis, percentage of observations with ≥64-knot sustained winds on the y-axis, and line color represents the center's **west/east-of-65°W longitude group**. Hover gives event-position counts and denominators for each year/group. This compares timing **and** a broad place classification. **Figure 4 — animated point map:** Center latitude/longitude define each plotted point, color indicates below/at-least-64-knot wind, the frame selects year, and hover gives exact storm, date, wind in knots, and center pressure in millibars. The map shows positions, while the line plot aggregates them into annual rates. An animation frame counts as one figure.

In [17]:
# Figure 3 — annual share of hurricane-force track points by longitude group
fig3 = px.line(
    cyclone_summary,
    x="year", y="force_percentage", color="longitude_band", markers=True,
    title="Atlantic cyclone positions with winds ≥64 kt by region and year (2020–2025)",
    labels={"year": "Year (UTC)", "force_percentage": "Positions with winds ≥64 kt (%)",
            "longitude_band": "Center longitude group"},
    hover_data={"hurricane_force_positions": True, "total_positions": True}
)
fig3.update_layout(width=950, height=520)
fig3.update_xaxes(dtick=1)
fig3.update_yaxes(range=[0, 60])
fig3.write_html("figures/atlantic_cyclone_yearly.html", include_plotlyjs=True)
fig3.show()

**What does Figure 1 show? Describe a pattern and record at least one value with its place, time, and units or count.**

Figure 3 shows that the **share of recorded Atlantic cyclone positions** with maximum sustained winds **≥64 knots** varied across years and geographic groups. In **2025**, **65/119 positions (54.6%)** west of **65°W** met the threshold, versus **36/140 (25.7%)** at or east of 65°W. By comparison, in **2021**, the western group had **33/189 (17.5%)**, whereas the eastern group had **84/237 (35.4%)**. Thus, the higher group changes across years. These are percentages of position records; repeated readings of a single storm contribute repeatedly.

In [18]:
# Figure 4 — the individual center positions behind the annual percentages
# Each frame is one year, color shows a >=64 knot observation.
fig4 = px.scatter_map(
    cyclone_analysis,
    lat="latitude", lon="longitude", color="wind_group",
    animation_frame="year", hover_name="storm_name",
    hover_data={"storm_id": True, "observation_time": True,
                "maximum_sustained_wind_knots": True,
                "minimum_central_pressure_mb": True,
                "latitude": ":.1f", "longitude": ":.1f", "wind_group": False},
    center={"lat": 25, "lon": -65}, zoom=2.8,
    map_style="open-street-map",
    title="Atlantic cyclone best-track center positions (2020–2025)",
    labels={"wind_group": "Wind category",
            "maximum_sustained_wind_knots": "Maximum sustained wind (kt)",
            "minimum_central_pressure_mb": "Central pressure (mb)"},
    color_discrete_map={"At least 64 kt": "#c92332", "Below 64 kt": "#2366b7"},
    category_orders={"year": sorted(cyclone_analysis["year"].unique())}
)
fig4.update_layout(width=950, height=650)
fig4.write_html("figures/atlantic_cyclone_map.html", include_plotlyjs=True)
fig4.show()

**What does Figure 2 add? Record at least one value with its place, time, and units or count, and compare its message with Figure 1.**

Figure 4 identifies **where and when** the observations behind the percentages occurred. For example, **Hurricane Milton** had a recorded maximum sustained wind of **155 knots**, center pressure **895 mb**, at approximately **21.8°N, 90.9°W**, on **October 7, 2024 at 20:00 UTC**. This position lies west of 65°W and exceeds the hurricane-force threshold, consistent with the western-group observations summarized in Figure 3. The map displays individual storm centers, whereas Figure 3 shows the share of best-track observations in each year/longitude group. A high dot does not indicate the physical storm's size or local impacts at that point.

### Step 4: Check

Use hover to compare at least two plotted values with the source records or a calculated summary. Check chronological order, location matching, and one unusual high, low, or apparent gap that matters to your question. Explain any problem you corrected. Do not repeat the initial missing-value count.

In [19]:
# Verify plotted rates and a geographic record directly against source rows
row_2025_west = cyclone_summary.loc[
    (cyclone_summary["year"] == 2025) &
    (cyclone_summary["longitude_band"] == "West of 65°W")
].iloc[0]
row_2021_west = cyclone_summary.loc[
    (cyclone_summary["year"] == 2021) &
    (cyclone_summary["longitude_band"] == "West of 65°W")
].iloc[0]

assert (int(row_2025_west["hurricane_force_positions"]),
        int(row_2025_west["total_positions"])) == (65, 119)
assert (int(row_2021_west["hurricane_force_positions"]),
        int(row_2021_west["total_positions"])) == (33, 189)

milton = cyclone_df.loc[
    (cyclone_df["storm_name"] == "Milton") &
    (cyclone_df["observation_time"] == pd.Timestamp("2024-10-07 20:00:00"))
]
assert len(milton) == 1
assert int(milton.iloc[0]["maximum_sustained_wind_knots"]) == 155
assert float(milton.iloc[0]["longitude"]) == -90.9
assert list(sorted(cyclone_analysis["year"].unique())) == list(range(2020, 2026))
assert cyclone_analysis[["latitude", "longitude"]].notna().all().all()

print("2025 west rate:", int(row_2025_west["hurricane_force_positions"]), "/",
      int(row_2025_west["total_positions"]), "=",
      round(float(row_2025_west["force_percentage"]), 1), "%")
print("2021 west rate:", int(row_2021_west["hurricane_force_positions"]), "/",
      int(row_2021_west["total_positions"]), "=",
      round(float(row_2021_west["force_percentage"]), 1), "%")
print("Verified 2024 storm point:")
print(milton[["storm_name", "observation_time", "latitude", "longitude",
              "maximum_sustained_wind_knots", "minimum_central_pressure_mb"]].to_string(index=False))
print("Years and source coordinates verified.")

2025 west rate: 65 / 119 = 54.6 %
2021 west rate: 33 / 189 = 17.5 %
Verified 2024 storm point:
storm_name    observation_time  latitude  longitude  maximum_sustained_wind_knots  minimum_central_pressure_mb
    Milton 2024-10-07 20:00:00      21.8      -90.9                           155                          895
Years and source coordinates verified.


**Which two values did you check, and did they agree? What did your check establish about order, location, or an unusual pattern?**

I verified **2025 west-of-65°W: 65 of 119 = 54.6%** and **2021 west-of-65°W: 33 of 189 = 17.5%** by matching the line-chart inputs to the original best-track rows. I also checked the geographic hover record for **Milton** on **October 7, 2024, 20:00 UTC**: **155 knots** at **21.8°N, 90.9°W**, consistent with the source CSV. The checks confirm the annual ordering from 2020 to 2025 and that the extreme high-wind record lies in the western longitude group. The changing annual rate is not itself proof that more individual storms occurred in one group; some storms have more observations than others.

### Step 5: Conclusion

Restate your research question, then answer it using evidence from **both** figures. Include a comparison across places and a comparison across times, with values and units. Explain whether the evidence supports your prediction.

**Restate your question and write your conclusion.**

**Question:** How did the share of Atlantic tropical cyclone best-track observations with maximum sustained wind **≥64 knots** differ by **year** and by centers west versus east of **65°W** during **2020–2025**? **Answer:** The record-based share changed substantially across both dimensions. In **2021**, the western group had **17.5%** (33/189), below the eastern group's **35.4%** (84/237). In **2025**, the western group reached **54.6%** (65/119), above the eastern group's **25.7%** (36/140). Figure 3 makes this time-and-region reversal clear; Figure 4 places individual records, such as **Milton at 155 knots on October 7, 2024** near 21.8°N, 90.9°W, on the geographic map. The evidence supports the prediction that the percentage of high-wind *positions* would vary by both year and longitude region. It does **not** show the percentage of all storms that reached hurricane strength.

### Step 6: Limitation

Identify **two limitations of your analysis**. Then explain which aspects of the research question are easier or harder to answer from each figure.

**What are two limitations, and how do the strengths and weaknesses of your figures affect the answer?**

**Two limitations:** (1) Best-track observations are **repeated positions of the same storms**, typically separated by several hours, with extra times for some events. The percent of recorded positions ≥64 knots is sensitive to storm duration and record frequency; it is **not** a count or proportion of unique hurricanes, and it cannot by itself establish risk to residents. (2) Splitting the basin at **65°W** is an arbitrary descriptive boundary: storm tracks cross it, and proximity to a coastline or wind field size is not represented. The line chart makes annual and west/east comparisons straightforward but hides individual tracks; the animated map provides coordinates, times, and storm examples but overlapping points and year aggregation obscure precise paths. Neither figure establishes impacts or causal reasons for differences.

## AI-use reflection

**Did you use AI? If so, describe one suggestion you checked against your source, code, or output and what you accepted, modified, or rejected. If not, state that you did not use it.**

**Yes.** ChatGPT assisted with drafting the analysis code, visualization choices, written explanations, and review checks. The assistant originally suggested a map using the deprecated Plotly `scatter_mapbox` function; this version uses the supported `px.scatter_map` instead. The notebook includes explicit code checks of source-record locations, dates, concentrations, and sample counts (e.g., Florida August's **191/1,236** observations and the two selected cyclone year/region counts); these checks should be rerun and the visual hover values personally reviewed before submission. The conclusions were limited to sampled observations and best-track positions instead of making unsupported claims about unsampled areas or storm impacts. The student is responsible for understanding, checking, and submitting this work.

## Submission checklist

- Include two distinct datasets, each with both usable spatial and temporal information, plus any lookup or boundary files.
- Document both sources, feature definitions, units, collection context, coverage, and download filters.
- Complete the feature table and every written response for both investigations.
- Ask one research question per dataset and complete all six workflow steps for each question.
- Include four finished figures: two different chart types for each question, with each pair addressing both space and time.
- Explain filtering, missing values, aggregations, row counts, and any changes in the unit of observation.
- Check plotted values, temporal order, and geographic interpretation against the data.
- Restate each question in its conclusion and support the answer with evidence from both figures.
- Identify two limitations per investigation and compare what the two displays reveal or obscure.
- Complete the AI-use reflection. Restart the kernel, run cells in order, and ensure interactive figures display correctly before submitting.